In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

# Classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import CategoricalNB

# Verification and Summary
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Pickling
import pickle

# JSON
import json

## Classification

In [ ]:
sig_to_evttagid = pd.read_csv('Inputs/sig-evttags-map-02.csv')
display(sig_to_evttagid.head(5))

df = pd.read_csv('Inputs/pmu_transmission_features.csv')
display(df.head(5))

sec_to_tert_dict = json.load(open('Inputs/sec_to_tert_dict_transmission.json'))

df_dist = pd.read_csv('Inputs/pmu_distribution_features.csv')
df_dist_2_3 = df_dist.loc[df_dist['sigid'] < 655].copy()
df_dist_4 = df_dist.loc[df_dist['sigid'] >= 655].copy().loc[:, ~df_dist.columns.isin(['mean_v', 'variance_v', 'skewness_v', 'kurtosis_v'])]
display(df_dist.head(5))

In [ ]:
# data_source = ['Transmission_PMU', 'Distribution_PMU_2_3', 'Distribution_PMU_4', 'POW', 'Lab_Testing']
# classifier = ['NB', 'RF', 'SVC' (which is RBF, i.e. default for sklearn), SVC_Linear', 'SVC_Polynomial']
def successiveBinaryClassification(test_features, label_list, data_source, classifier):
    proba_temp = 0
    pred_class = ''
    
    for evttag in label_list:
        fname = classifier + '_' + evttag
        # Change file
        with open(f'Classifiers/{data_source}/{classifier}_{evttag}.pickle', 'rb') as handle:
            globals()[fname] = pickle.load(handle)
            
        if classifier == 'NB':
            scaler = MinMaxScaler()
            test_features = scaler.fit_transform(test_features)
            
        proba = globals()[fname].predict_proba(test_features)
        if proba[0][1] > proba_temp:
            proba_temp = proba[0][1]
            pred_class = evttag
    
    return pred_class

In [ ]:
primary_arr = ['Events', 'State', 'Equipment', 'Conditions', 'Phase']
acc = []

for primary_event in primary_arr:
    print(f'Classifying for testing dataset for {primary_event}...\n')
    df1 = df.copy().loc[df['primary'] == primary_event].dropna().reset_index()

    # Unique tertiary labels ordered from highest count number to lowest count number
    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    features = df1.iloc[:, -24:-1]
    label = df1['secondary']

    x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)
    # Classify each entry in the testing dataset individually and print out the RF classifier
    # 'True' label that outputs the highest probability.
    actual_class_sec = []
    pred_class_sec = []
    actual_class_tert = []
    pred_class_tert = []

    for idx in x_test.index:
#         print(f'Classifying secondary label for row {idx}')
        test_features = x_test[x_test.index == idx]
        pred_class = successiveBinaryClassification(test_features, arr, 'Transmission_PMU', 'RF')
        actual_class_sec.append(df1.iloc[idx]['secondary'])
        pred_class_sec.append(pred_class)

#         print(f'Classifying tertiary label for row {idx}\n')
        pred_class = successiveBinaryClassification(test_features, sec_to_tert_dict[pred_class], 'Transmission_PMU', 'RF')
        actual_class_tert.append(df1.iloc[idx]['tertiary'])
        pred_class_tert.append(pred_class)


    test_results = pd.DataFrame(columns=['Actual_Secondary', 'Predicted_Secondary', 
                                         'Actual_Tertiary', 'Predicted_Tertiary'])
    test_results['Actual_Secondary'] = actual_class_sec
    test_results['Predicted_Secondary'] = pred_class_sec
    test_results['Actual_Tertiary'] = actual_class_tert
    test_results['Predicted_Tertiary'] = pred_class_tert
    
    correct_classification_sec = test_results.loc[test_results['Actual_Secondary'] == test_results['Predicted_Secondary']].shape[0]
    correct_classification_tert = test_results.loc[test_results['Actual_Tertiary'] == test_results['Predicted_Tertiary']].shape[0]
    acc.append(correct_classification_tert / test_results.shape[0] * 100)
    
results = pd.DataFrame(columns=['primary', 'acc'])
results['primary'] = primary_arr
results['acc'] = acc
results

## No need to re-run below unless new classifiers must be trained. 

In [ ]:
# Insert which df has features and statistical measures, then run all the code blocks below
df_master = df

In [ ]:
# Make a dictionary that stores an array of tertiary labels for each secondary label.
# This is for transmission PMU.
sec_to_tert_dict = {}

primary_arr = ['Events', 'State', 'Equipment', 'Conditions', 'Phase']

for primary_event in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_event].dropna().reset_index()

    # Unique tertiary labels ordered from highest count number to lowest count number
    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    for evttag in arr:
        temp = df_master.copy().loc[df_master['secondary'] == evttag].dropna().reset_index()
        temp_arr = (np.asanyarray(temp.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)).tolist()
        sec_to_tert_dict[evttag] = temp_arr
        
print(sec_to_tert_dict)
    
# # ============= CHANGE SAVE FILE NAME ============= #
# with open('sec_to_tert_dict_provider_4.json', 'w') as outfile:
#     json.dump(sec_to_tert_dict, outfile)

### Naïve (Categorical) Bayesian
Important to note that MinMaxScaler is used because the NB classifier does not take negative values in feature set.

In [ ]:
for primary_events in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_events].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr:
            print(f'Training (Secondary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['NB_%s' % evttag] = CategoricalNB()
            scaler = MinMaxScaler()
            x_train = scaler.fit_transform(x_train)
            globals()['NB_%s' % evttag].fit(x_train, y_train_copy.values)
            
            print(f'Pickling NB classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/NB_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['NB_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

        # ============= TERTIARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['tertiary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr_tertiary:
            print(f'Training (Tertiary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['NB_%s' % evttag] = CategoricalNB()
            scaler = MinMaxScaler()
            x_train = scaler.fit_transform(x_train)
            globals()['NB_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling NB classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/NB_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['NB_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

### Support Vector Machine
Need to set probability=True so that probabilities will be outputted by classifier. 

In [ ]:
kernel = 'linear'
for primary_events in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_events].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr:
            print(f'Training (Secondary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['SVC_Linear_%s' % evttag] = SVC(kernel=kernel, probability=True)
            globals()['SVC_Linear_%s' % evttag].fit(x_train, y_train_copy.values)
            
            print(f'Pickling SVC classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/SVC_Linear_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['SVC_Linear_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

        # ============= TERTIARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['tertiary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr_tertiary:
            print(f'Training (Tertiary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['SVC_Linear_%s' % evttag] = SVC(kernel=kernel, probability=True)
            globals()['SVC_Linear_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling SVC classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/SVC_Linear_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['SVC_Linear_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

### Random Forest

In [ ]:
n_estimators = 1000
max_depth = 8

for primary_event in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_event].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr:
            print(f'Training (Secondary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['RF_%s' % evttag] = RandomForestClassifier(n_estimators=n_estimators, 
                                                                 max_depth=max_depth, random_state=42)
            globals()['RF_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling RF classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/RF_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['RF_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

        # ============= TERTIARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['tertiary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr_tertiary:
            print(f'Training (Tertiary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['RF_%s' % evttag] = RandomForestClassifier(n_estimators=n_estimators, 
                                                                 max_depth=max_depth, random_state=42)
            globals()['RF_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling RF classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/RF_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['RF_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)